# Tugas Pengganti IT Incubation – Machine Learning SDG 4

**Topik:** Prediksi Kelulusan Siswa  
**SDG:** SDG 4 – Pendidikan Berkualitas  
**Task:** Klasifikasi  
**Algoritma:** Logistic Regression (hanya 1 algoritma)

> Dataset adalah **data simulasi** untuk pembelajaran, bukan data siswa nyata.


## 1. Import library

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib


## 2. Membaca dataset

Jika dijalankan di Google Colab dan file belum ada, sel berikut akan membuka pilihan upload. Jika file sudah berada satu folder dengan notebook, file akan langsung dibaca.

In [ ]:
DATASET = "dataset_siswa_sdg4.csv"

if not os.path.exists(DATASET):
    try:
        from google.colab import files
        uploaded = files.upload()
        DATASET = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "dataset_siswa_sdg4.csv tidak ditemukan. Letakkan dataset di folder yang sama dengan notebook."
        )

data = pd.read_csv(DATASET)
print("File yang digunakan:", DATASET)
display(data.head())


## 3. Memahami dataset

In [ ]:
print("Ukuran data:", data.shape)
print("\nTipe data:")
print(data.dtypes)
print("\nJumlah nilai kosong:")
print(data.isnull().sum())
print("\nDistribusi target:")
print(data["Lulus"].value_counts())


## 4. Preprocessing

- `ID_Siswa` tidak digunakan karena hanya identitas.
- Target `Lulus` diubah menjadi angka: **Ya = 1** dan **Tidak = 0**.
- Data dibagi menjadi **80% training** dan **20% testing**.
- Fitur numerik dinormalisasi dengan `StandardScaler`.


In [ ]:
features = [
    "Kehadiran_Persen",
    "Jam_Belajar_per_Hari",
    "Nilai_Tugas",
    "Nilai_Ujian"
]

X = data[features]
y = data["Lulus"].map({"Tidak": 0, "Ya": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data training:", X_train.shape)
print("Data testing :", X_test.shape)


## 5. Training – Logistic Regression

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("Training selesai.")


## 6. Evaluasi model

In [ ]:
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Tidak Lulus", "Lulus"]
))

print("Confusion Matrix:")
print(cm)


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix – Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Tidak Lulus", "Lulus"])
plt.yticks([0, 1], ["Tidak Lulus", "Lulus"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


## 7. Menyimpan model untuk Streamlit

In [ ]:
joblib.dump(model, "model_logistic_regression.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model dan scaler berhasil disimpan.")


## 8. Kesimpulan

Model Logistic Regression digunakan untuk mengklasifikasikan status kelulusan berdasarkan kehadiran, jam belajar, nilai tugas, dan nilai ujian. Pada pembagian data `random_state=42`, model memperoleh **accuracy 80%** pada 30 data testing.

Karena dataset bersifat simulasi, hasil ini hanya untuk demonstrasi proses Machine Learning dan tidak boleh dianggap sebagai penilaian siswa nyata.
